# Agentic RAG Foundations: Intelligent Retrieval Patterns

### What You'll Learn

Building directly on **6.AdvancedAgents** (memory, streaming, observability), this notebook introduces **foundational Agentic RAG patterns** where agents make intelligent decisions about information retrieval rather than just using retrieval as a simple tool.

**Prerequisites:** 
- Complete **6.AdvancedAgents** - Understanding of agent memory, streaming, and observability
- Familiar with basic RAG concepts from earlier notebooks

**Core Agentic RAG Foundations:**

1. **Query Planning & Analysis** - Agents analyze queries before retrieving
2. **Smart Retrieval Strategies** - Adaptive retrieval based on query type
3. **Document Relevance Grading** - Agents evaluate retrieval quality
4. **Self-Correcting RAG Loops** - Agents improve answers through iteration
5. **RAG with Agent Memory** - Context-aware retrieval using conversation history

### From Advanced Agents to Agentic RAG

**6.AdvancedAgents taught us:**
- Agents with persistent memory across conversations
- Streaming responses for better user experience  
- Error handling and observability for production systems
- **Agents that reason about their actions**

**Agentic RAG Foundations (This Notebook):**
- ✅ **Apply agent reasoning to RAG** - Not just "retrieve and answer"
- ✅ **Query Planning** - Agents decide HOW to retrieve before retrieving
- ✅ **Quality Assessment** - Agents grade their own retrieval results
- ✅ **Self-Correction** - Agents retry with better strategies when needed
- ✅ **Memory Integration** - Use conversation context for smarter retrieval

### Why This Matters

**Traditional RAG:** Query → Retrieve → Generate (static pipeline)

**Basic RAG + Agents:** Query → Agent chooses retrieval tool → Generate

**Agentic RAG Foundations:** 
```
Query → Agent plans retrieval strategy → Retrieves → Grades results → 
Self-corrects if needed → Generates contextual answer
```

### Real-World Impact

These patterns enable:
- **Self-Improving Q&A Systems** - Get better at retrieval over time
- **Context-Aware Help Desks** - Remember what users asked before
- **Adaptive Research Tools** - Adjust retrieval strategies per query type
- **Quality-Controlled RAG** - Ensure retrieved content is actually relevant

### Learning Path

After mastering these foundations, you'll be ready for:
- **8.MultiSourceRAG** - Routing between multiple knowledge bases
- **9.AgentSupervisor** - Orchestrating multiple specialized agents

---

Let's build RAG systems that think before they retrieve! 🧠📚

## Bridge from Advanced Agents: Applying Agent Reasoning to RAG

In **6.AdvancedAgents**, you learned to build sophisticated agents with:
- **Memory systems** for conversation context
- **Streaming responses** for real-time feedback
- **Error handling** for robust operation
- **Observability** to monitor agent behavior

**Now we apply these same patterns to RAG systems.** Instead of agents just "using retrieval as a tool," we create agents that:

![agenticrag.png](../../Assets/images/agenticrag.png)

### 🧠 **Think About Retrieval Strategy**
```python
# Traditional RAG
query = "How do I handle errors in Python?"
docs = retriever.retrieve(query)  # Static retrieval
answer = llm.generate(docs + query)

# Agentic RAG 
query = "How do I handle errors in Python?"
agent.plan_retrieval(query)      # 🧠 Agent analyzes the query
docs = agent.smart_retrieve()    # 🎯 Agent chooses best strategy  
quality = agent.grade_docs(docs) # ✅ Agent evaluates results
if quality.poor:
    docs = agent.retry_retrieval() # 🔄 Agent self-corrects
answer = agent.generate_with_memory(docs, query) # 💭 Uses conversation context
```

### 🎯 **Core Learning Progression**

1. **Query Analysis** - Understand what type of information is needed
2. **Strategic Retrieval** - Choose the right retrieval approach  
3. **Quality Assessment** - Evaluate if retrieved content is relevant
4. **Self-Correction** - Retry with different strategies when needed
5. **Memory Integration** - Use conversation history for better context

Let's start building! 🚀

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
]

docs = [WebBaseLoader(url).load() for url in urls]

In [ ]:
docs[0][0].page_content.strip()[:1000]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)

In [ ]:
doc_splits[0].page_content.strip()

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

## **2. Create a Retriever Tool**


In [ ]:
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_blog_posts",
    "Search and return information about Lilian Weng blog posts.",
)

In [ ]:
retriever_tool.invoke({"query": "types of reward hacking"})

## **3. Generate Query**

In [ ]:
from langgraph.graph import MessagesState
from langchain.chat_models import init_chat_model

response_model = init_chat_model("openai:gpt-4.1", temperature=0)


def generate_query_or_respond(state: MessagesState):
    """Call the model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply respond to the user.
    """
    response = (
        response_model
        .bind_tools([retriever_tool]).invoke(state["messages"])
    )
    return {"messages": [response]}

In [ ]:
input = {"messages": [{"role": "user", "content": "hello!"}]}
generate_query_or_respond(input)["messages"][-1].pretty_print()

In [ ]:
input = {
    "messages": [
        {
            "role": "user",
            "content": "What does Lilian Weng say about types of reward hacking?",
        }
    ]
}
generate_query_or_respond(input)["messages"][-1].pretty_print()

## Step 4: Document Grading - How the LLM Reasons About Relevance

### 🧠 **Understanding the Agent's Decision-Making Process**

When the agent retrieves documents, it doesn't just accept them blindly. Instead, it acts as an **intelligent quality controller** that evaluates whether the retrieved content actually helps answer the user's question.

**Here's how the LLM reasons through document grading:**

### 🔍 **The Agent's Internal Reasoning Process**

```
Agent Thinking Process:
1. "I retrieved this document for the user's question"
2. "Let me analyze: Does this document contain relevant information?"
3. "I'll look for keywords, concepts, or semantic meaning that relates to the question"
4. "Based on my analysis, I'll give a binary decision: relevant or not relevant"
5. "If relevant → proceed to generate answer"
6. "If not relevant → I need to rewrite the question and try again"
```

### 📋 **What the LLM Evaluates**

The grading LLM considers:
- **Keyword Matching**: Does the document contain terms from the question?
- **Semantic Relevance**: Does the document discuss the same concepts/topics?
- **Contextual Meaning**: Could this information help answer the question?
- **Completeness**: Does the document provide sufficient detail to form an answer?

**Example Decision Flow:**
```
Question: "What does Lilian Weng say about reward hacking?"
Retrieved Document: "reward hacking can be categorized into two types..."

LLM Reasoning:
✅ "This document mentions 'reward hacking' (keyword match)"
✅ "It provides categorization information (semantic relevance)" 
✅ "This directly answers what is being asked (contextual meaning)"
→ Decision: "yes" (relevant) → Generate answer

Vs.

Retrieved Document: "meow"
LLM Reasoning:
❌ "No keywords related to reward hacking"
❌ "No semantic connection to the question"
❌ "Cannot help answer the question"
→ Decision: "no" (not relevant) → Rewrite question and retry
```

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

GRADE_PROMPT = (
    "You are a grader assessing relevance of a retrieved document to a user question. \n "
    "Here is the retrieved document: \n\n {context} \n\n"
    "Here is the user question: {question} \n"
    "If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n"
    "Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."
)


class GradeDocuments(BaseModel):
    """Grade documents using a binary score for relevance check."""

    binary_score: str = Field(
        description="Relevance score: 'yes' if relevant, or 'no' if not relevant"
    )


grader_model = init_chat_model("openai:gpt-4.1", temperature=0)


def grade_documents(
    state: MessagesState,
) -> Literal["generate_answer", "rewrite_question"]:
    """Determine whether the retrieved documents are relevant to the question."""
    question = state["messages"][0].content
    context = state["messages"][-1].content

    prompt = GRADE_PROMPT.format(question=question, context=context)
    response = (
        grader_model
        .with_structured_output(GradeDocuments).invoke(
            [{"role": "user", "content": prompt}]
        )
    )
    score = response.binary_score

    if score == "yes":
        return "generate_answer"
    else:
        return "rewrite_question"

### 🎯 **How the Grading Function Routes the Agent's Next Action**

**The `grade_documents` function acts as a smart router:**

```python
# The agent's decision tree:
if score == "yes":  # Document is relevant
    return "generate_answer"  # → Go directly to answer generation
else:  # Document is not relevant  
    return "rewrite_question"  # → Improve the question and try again
```

**This creates a self-correcting loop:**
1. **Retrieve documents** based on original question
2. **Grade quality** of retrieved content
3. **Route intelligently**:
   - **Good retrieval** → Generate answer immediately
   - **Poor retrieval** → Self-correct by rewriting question

**Why this matters:** Traditional RAG would generate an answer even with irrelevant documents, leading to hallucinations or poor responses. Agentic RAG **quality-checks** first!

### 🔄 **The Self-Correction Advantage**

```
Traditional RAG: Question → Retrieve → Generate (even with bad docs)
Agentic RAG: Question → Retrieve → Grade → [Route to correction if needed] → Generate
```

Let's see this in action:

In [ ]:
from langchain_core.messages import convert_to_messages

input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {"role": "tool", "content": "meow", "tool_call_id": "1"},
        ]
    )
}
grade_documents(input)

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering",
                "tool_call_id": "1",
            },
        ]
    )
}
grade_documents(input)

## Step 5: Question Rewriting - How the Agent Improves Its Search Strategy

### 🔄 **When Document Grading Fails: The Agent's Self-Improvement Process**

When the document grader returns "no" (documents aren't relevant), the agent doesn't give up. Instead, it **reasons about why the retrieval failed** and **improves its search strategy**.

### 🧠 **The LLM's Question Rewriting Reasoning Process**

```
Agent's Self-Reflection Process:
1. "My retrieval didn't work well - the documents weren't relevant"
2. "Let me analyze the original question: What might be causing poor retrieval?"
3. "Could the question be too vague? Too specific? Using wrong terminology?"
4. "I'll reformulate to be clearer, more specific, or use better search terms"
5. "This improved question should retrieve more relevant documents"
```

### 🎯 **What the Question Rewriter Considers**

The rewriting LLM analyzes:
- **Semantic Intent**: What is the user really trying to learn?
- **Search Terminology**: Are there better keywords for retrieval?
- **Question Clarity**: Is the question too ambiguous?
- **Specificity Level**: Does it need more or less detail?

### 📝 **Example Rewriting Reasoning**

**Original Question:** "What does Lilian Weng say about types of reward hacking?"

**LLM's Rewriting Analysis:**
```
Agent Reasoning:
🔍 "The user wants specific information from Lilian Weng about reward hacking types"
🎯 "My retrieval failed, so maybe I need to be more specific about the categorization"
💡 "I should focus on the classification/taxonomy aspect"
✨ "Better search terms might include: 'categorization', 'classification', 'taxonomy'"

Improved Question: "How does Lilian Weng categorize or classify different types of reward hacking in machine learning?"
```

**Why this works better:**
- More specific search intent
- Better terminology for retrieval
- Clearer focus on the desired information type

In [ ]:
REWRITE_PROMPT = (
    "Look at the input and try to reason about the underlying semantic intent / meaning.\n"
    "Here is the initial question:"
    "\n ------- \n"
    "{question}"
    "\n ------- \n"
    "Formulate an improved question:"
)


def rewrite_question(state: MessagesState):
    """Rewrite the original user question."""
    messages = state["messages"]
    question = messages[0].content
    prompt = REWRITE_PROMPT.format(question=question)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [{"role": "user", "content": response.content}]}

### 🔄 **The Complete Self-Correction Loop in Action**

**Understanding the Agent's Improvement Cycle:**

```
Step 1: Original Question + Retrieval
"What does Lilian Weng say about types of reward hacking?"
↓ (retrieve documents)
Retrieved: "meow" (irrelevant content)
↓ (grade documents)
Grade: "no" (not relevant)
↓ (route decision)
Route: "rewrite_question"

Step 2: Question Analysis + Rewriting  
Agent analyzes: "My search wasn't specific enough"
Agent improves: "How does Lilian Weng categorize different types of reward hacking?"
↓ (retrieve again with better question)
Retrieved: "reward hacking can be categorized into two types..."
↓ (grade documents again)
Grade: "yes" (relevant!)
↓ (route decision)
Route: "generate_answer"

Step 3: Generate Final Answer
Agent synthesizes relevant information into response
```

### 🎯 **Key Insight: Agent Learning**

The agent demonstrates **meta-reasoning** - it reasons about its own reasoning process:
- **Self-awareness**: "My retrieval strategy didn't work"
- **Self-analysis**: "Why did it fail? How can I improve?"
- **Self-correction**: "Let me try a better approach"
- **Self-validation**: "Did my improvement work better?"

This is fundamentally different from static RAG pipelines!

Let's test this rewriting process:

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {"role": "tool", "content": "meow", "tool_call_id": "1"},
        ]
    )
}

response = rewrite_question(input)
print(response["messages"][-1]["content"])

## Step 6: Answer Generation - The Final Reasoning Step

### ✅ **When the Agent Finally Has Quality Information**

After the agent successfully retrieves and validates relevant documents, it moves to the final reasoning step: **synthesizing a high-quality answer**.

### 🧠 **The Answer Generation Reasoning Process**

```
Agent's Final Reasoning:
1. "I now have relevant, validated documents"
2. "Let me extract the key information that answers the user's question"
3. "I'll synthesize this into a clear, concise response"
4. "I'll stay grounded in the retrieved content and avoid hallucination"
5. "If the information is insufficient, I'll honestly say I don't know"
```

### 🎯 **Quality-Controlled Answer Generation**

Unlike traditional RAG, the agent only generates answers when it has **verified relevant content**:

**Traditional RAG:**
```
Question → Retrieve anything → Generate (even with bad docs)
Result: Potentially hallucinated or irrelevant answers
```

**Agentic RAG:**
```
Question → Retrieve → Grade quality → [Self-correct if needed] → Generate with confidence
Result: High-quality, grounded, relevant answers
```

### 📋 **The Generation LLM's Constraints**

The answer generator follows strict guidelines:
- **Stay grounded**: Only use information from retrieved documents
- **Be concise**: Maximum three sentences for clarity
- **Be honest**: Say "I don't know" if information is insufficient
- **Be relevant**: Directly address the user's specific question

This ensures every generated answer is **trustworthy and accurate**!

In [ ]:
GENERATE_PROMPT = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \n"
    "Context: {context}"
)


def generate_answer(state: MessagesState):
    """Generate an answer."""
    question = state["messages"][0].content
    context = state["messages"][-1].content
    prompt = GENERATE_PROMPT.format(question=question, context=context)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [response]}

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering",
                "tool_call_id": "1",
            },
        ]
    )
}

response = generate_answer(input)
response["messages"][-1].pretty_print()

## Step 7: Assemble the Graph - Building the Agent's Decision Tree

### 🏗️ **What Graph Assembly Does**

The graph assembly creates a **computational workflow** that connects all our intelligent components into a unified agentic system. Think of it as building the "brain architecture" for our RAG agent.

### 🧠 **The Graph Represents the Agent's Decision-Making Flow**

```
📝 User Question
    ↓
🤔 Generate Query or Respond (decides: retrieve or answer directly?)
    ↓
🔄 Conditional Routing:
    • If needs retrieval → Go to Retrieve
    • If can answer directly → End with response
    ↓
📚 Retrieve Documents
    ↓
✅ Grade Documents (quality check)
    ↓
🔄 Conditional Routing Based on Quality:
    • If documents are good → Generate Answer → End
    • If documents are poor → Rewrite Question → Start over
```

### 🎯 **Key Benefits of Using LangGraph**

**1. 📊 Visual Agent Logic**
- **Clear decision paths**: Every routing choice is explicit
- **Debuggable workflows**: You can see exactly where the agent goes
- **Visual representation**: The graph shows the complete flow

**2. 🔄 Self-Correcting Loops**
- **Built-in retry logic**: Poor results automatically trigger improvements
- **State management**: The graph maintains conversation context
- **Conditional routing**: Smart decisions at every step

**3. 🎮 Fine-Grained Control**
- **Custom routing logic**: You define exactly how decisions are made
- **State tracking**: Monitor what the agent is thinking at each step
- **Modular components**: Easy to modify or extend individual parts

**4. 🚀 Production Benefits**
- **Scalable architecture**: Handles complex multi-step reasoning
- **Error handling**: Built-in fallback and retry mechanisms
- **Observability**: Full visibility into agent decision-making

### 🔧 **What Each Component Does**

```python
# The nodes (thinking steps):
workflow.add_node(generate_query_or_respond)  # 🤔 Initial reasoning
workflow.add_node("retrieve", ToolNode(...))  # 📚 Document retrieval
workflow.add_node(rewrite_question)          # 📝 Question improvement
workflow.add_node(generate_answer)           # ✅ Final answer generation

# The edges (decision paths):
workflow.add_conditional_edges(...)          # 🔄 Smart routing logic
```

**Without LangGraph:** Linear, static RAG pipeline
**With LangGraph:** Intelligent, adaptive, self-correcting agent system!

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

workflow = StateGraph(MessagesState)

# Define the nodes we will cycle between
workflow.add_node(generate_query_or_respond)
workflow.add_node("retrieve", ToolNode([retriever_tool]))
workflow.add_node(rewrite_question)
workflow.add_node(generate_answer)

workflow.add_edge(START, "generate_query_or_respond")

# Decide whether to retrieve
workflow.add_conditional_edges(
    "generate_query_or_respond",
    # Assess LLM decision (call `retriever_tool` tool or respond to the user)
    tools_condition,
    {
        # Translate the condition outputs to nodes in our graph
        "tools": "retrieve",
        END: END,
    },
)

# Edges taken after the `action` node is called.
workflow.add_conditional_edges(
    "retrieve",
    # Assess agent decision
    grade_documents,
)
workflow.add_edge("generate_answer", END)
workflow.add_edge("rewrite_question", "generate_query_or_respond")

# Compile
graph = workflow.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Step 8: Run the Agentic RAG - Watching the Agent Think

### 🎬 **What This Execution Shows**

When we run the agentic RAG system, we get to **watch the agent's thinking process** in real-time. Each update shows a step in the agent's reasoning journey from question to final answer.

### 🧠 **Understanding the Agent's Decision Journey**

**The streaming output reveals the agent's internal monologue:**

```
🎯 Step 1: "generate_query_or_respond"
Agent thinks: "I need to analyze this question about Lilian Weng and reward hacking.
Should I retrieve information or can I answer directly? I need to retrieve."

📚 Step 2: "retrieve" 
Agent thinks: "Let me search for documents about Lilian Weng and reward hacking types."
Action: Executes retrieval tool with search query

✅ Step 3: "grade_documents" (invisible but happening)
Agent thinks: "Are these retrieved documents relevant to the question?"
Decision: Routes to either "generate_answer" or "rewrite_question"

🎯 Step 4a: "generate_answer" (if documents were good)
Agent thinks: "Great! I have relevant information. Let me synthesize a response."

OR

📝 Step 4b: "rewrite_question" (if documents were poor)
Agent thinks: "These documents aren't helpful. Let me improve my search strategy."
→ Loops back to Step 1 with better question
```

### 🔍 **What Each Update Tells Us**

**"Update from node [node_name]"** shows:
- **Which reasoning step** the agent is currently executing
- **What decision** the agent just made
- **The agent's output** at that step (query, retrieved docs, final answer)

### 🎯 **Key Benefits of This Approach**

**1. 🕵️ Complete Transparency**
- See every decision the agent makes
- Understand why certain paths were chosen
- Debug issues when they occur

**2. 🔄 Adaptive Behavior**
- Watch the agent self-correct in real-time
- See how question rewriting improves results
- Observe the quality assessment process

**3. 📊 Educational Value**
- Students understand the agent's reasoning process
- Clear visualization of multi-step thinking
- Demonstrates the difference from static RAG

**4. 🚀 Production Monitoring**
- Track agent performance across steps
- Identify bottlenecks or failure points
- Monitor self-correction frequency

### 💡 **What Makes This "Agentic"**

Traditional RAG: **Question** → Retrieve → Generate **(done)**

Agentic RAG: **Question** → 🤔 Analyze → 📚 Retrieve → ✅ Grade → 🔄 Self-correct if needed → 🎯 Generate **(intelligent process)**

**You're about to see an agent that thinks, evaluates, and improves its own performance!**

In [ ]:
for chunk in graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            }
        ]
    }
):
    for node, update in chunk.items():
        print("Update from node", node)
        update["messages"][-1].pretty_print()
        print("\n\n")

## Conclusion: Key Concepts and Next Steps

🎯 **What We've Accomplished**

In this notebook, we built a sophisticated **Agentic RAG system** that demonstrates several advanced concepts:

### 🧭 **Intelligent Routing** 
- **Multi-Source Architecture**: Technical docs, business knowledge, troubleshooting guides
- **Smart Routing Decisions**: LLM-powered routing with confidence scoring
- **Memory-Enhanced Routing**: Uses historical patterns to improve decisions

### 🔄 **Self-Correcting Capabilities**
- **Quality Assessment**: Automated response evaluation and improvement suggestions
- **Adaptive Retrieval**: Adjusts strategy based on initial results
- **Iterative Refinement**: Retries with different approaches when needed


---